<table style="width: 100%;">
<tr>
<td style="width: 50%; text-align: right; vertical-align: middle;">
<img src="https://github.com/gitpizzanow/dummy-files/blob/main/tp3nlp.jpg?raw=true" width="150">
</td>
<td style="width: 50%; text-align: left; vertical-align: middle;">

##  (NLP)   | TF-IDF + Cosine Similarity
> [SERIE](https://tp3-nlp-ing4.netlify.app/)


* *Document Frequency (DF)*
* *IDF + Smoothing*
* *TF (Term Frequency)*
* *Cosine Similarity*



</td>
</tr>
</table>

>Data: Wikipedia-like dataset

In [1]:
from sklearn.datasets import fetch_20newsgroups

docs = fetch_20newsgroups(
    subset='train',
    remove=('headers', 'footers', 'quotes')
).data[:3000]

In [2]:
type(docs)

list

In [3]:
docs[10]

'I have a line on a Ducati 900GTS 1978 model with 17k on the clock.  Runs\nvery well, paint is the bronze/brown/orange faded out, leaks a bit of oil\nand pops out of 1st with hard accel.  The shop will fix trans and oil \nleak.  They sold the bike to the 1 and only owner.  They want $3495, and\nI am thinking more like $3K.  Any opinions out there?  Please email me.\nThanks.  It would be a nice stable mate to the Beemer.  Then I\'ll get\na jap bike and call myself Axis Motors!\n\n-- \n-----------------------------------------------------------------------\n"Tuba" (Irwin)      "I honk therefore I am"     CompuTrac-Richardson,Tx\nirwin@cmptrc.lonestar.org    DoD #0826          (R75/6)'

![STEP 1 - Preprocessing](https://img.shields.io/badge/STEP%201%20-%20Preprocessing-blue)

In [4]:

import re

def preprocess(text):

    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)
    tokens = text.split()
    return tokens

![STEP 2 - Vocabulary](https://img.shields.io/badge/STEP%202%20-%20Vocabulary-blue)

In [5]:

def build_vocab(docs):

    vocab = set()
    for doc in docs:
        tokens = preprocess(doc)
        vocab.update(tokens)
    return list(vocab)

[![STEP 3 DF](https://img.shields.io/badge/STEP_3_DF-Document_Frequency-pink)](https://digitalpro.dev)

In [6]:

def compute_df(docs, vocab):

    df = {word: 0 for word in vocab}
    for word in vocab:
        count = 0
        for doc in docs:
            tokens = preprocess(doc)
            if word in tokens:
                count += 1
        df[word] = count
    return df

[![STEP 4 IDF](https://img.shields.io/badge/STEP_4_IDF-Inverse_Document_Frequency-pink)](https://digitalpro.dev)

In [7]:

import numpy as np

def compute_idf(df, N):

    idf = {}
    for word, doc_freq in df.items():
        idf[word] = np.log((N + 1) / (doc_freq + 1)) + 1
    return idf

[![STEP 5 TF Vector](https://img.shields.io/badge/STEP_5_TF-Term_Frequency_Vector-pink)](https://digitalpro.dev)

In [8]:
def compute_tf(doc, vocab):

    tokens = preprocess(doc)
    total_words = len(tokens)
    tf = np.zeros(len(vocab))
    for i, word in enumerate(vocab):
        tf[i] = tokens.count(word) / total_words if total_words > 0 else 0
    return tf

[![STEP 6 TF-IDF Matrix](https://img.shields.io/badge/STEP_6_TFIDF-Build_TF_IDF_Matrix-pink)](https://digitalpro.dev)

In [9]:
def build_tfidf(docs):

    vocab = build_vocab(docs)
    N = len(docs)
    df = compute_df(docs, vocab)
    idf = compute_idf(df, N)

    tfidf_matrix = []
    for doc in docs:
        tf = compute_tf(doc, vocab)
        tfidf = np.array([tf[i] * idf[vocab[i]] for i in range(len(vocab))])
        tfidf_matrix.append(tfidf)

    return np.array(tfidf_matrix), vocab

[![STEP 7 Cosine Similarity](https://img.shields.io/badge/STEP_7-Cosine_Similarity-pink)](https://digitalpro.dev)

In [10]:
def cosine(a, b):

    dot = np.dot(a, b)
    norm_a = np.linalg.norm(a)
    norm_b = np.linalg.norm(b)
    if norm_a == 0 or norm_b == 0:
        return 0.0
    return dot / (norm_a * norm_b)

[![STEP 8 Search Engine](https://img.shields.io/badge/STEP_8-Search_Engine_REAL_VERSION-orange)](https://digitalpro.dev)

In [11]:
import numpy as np
def search(query, docs, top_k=5):
    tfidf_matrix, vocab = build_tfidf(docs)

    q_vec = np.zeros(len(vocab)) # build query vector
    q_words = preprocess(query)

    for i, w in enumerate(vocab):
        q_vec[i] = q_words.count(w)

    if np.sum(q_vec) > 0:
        q_vec = q_vec / np.sum(q_vec)

    scores = []

    for i, doc_vec in enumerate(tfidf_matrix):
        score = cosine(q_vec, doc_vec)
        scores.append((score, i))

    scores.sort(reverse=True)

    return scores[:top_k]

> TEST

In [ ]:
query = "machine learning neural network"
results = search(query, docs)

for score, idx in results:
    print(score)
    print(docs[idx][:200])
    print("-" * 50)